# 03 · Query-dependent pooling on a budget (Intermediate: limitation + initial idea)
Inference only (no training). Uses `runs/<EXP>/dump_model_best/dump_original.pt` and `runs/<EXP>/model_best.pth` of every finished seed.

Runtime: **A100** (timing table). Expected: about 10–20 minutes. Run ALL cells in order (cell 1 pulls the latest code; do not skip it).

In [ ]:
# ==== 0. Settings (edit if needed) ====
DRIVE_DIR  = "/content/drive/MyDrive/컴비 프로젝트"  # Google Drive folder (persistent: data copy, checkpoints, runs)
DRIVE_ROOT = "/content/cv_drive"                   # space-free alias (symlink) of DRIVE_DIR, used in all shell commands
REPO_URL   = "https://github.com/beblack127/computervision.git"
BRANCH     = "main"
REPO       = "/content/computervision"
LOCAL_DATA = "/content/data/msrvtt"                # fast local disk (wiped when runtime ends)
CACHE_DIR  = "/content/cache/msrvtt_u12"           # frame cache (uniform 12 frames, raw uint8)
NUM_WORKERS = 8                                    # official default; keep fixed across runs

from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, json, time
os.makedirs(DRIVE_DIR, exist_ok=True)
# The folder name contains a space, which would split unquoted paths in `!` shell commands -> use a symlink
if os.path.islink(DRIVE_ROOT) and os.readlink(DRIVE_ROOT) != DRIVE_DIR:
    os.remove(DRIVE_ROOT)
if not os.path.exists(DRIVE_ROOT):
    os.symlink(DRIVE_DIR, DRIVE_ROOT)
print("Drive folder:", DRIVE_DIR, "->", DRIVE_ROOT, "| contents:", sorted(os.listdir(DRIVE_ROOT)))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc && free -g | head -2 && df -h /content | tail -1

EXPS = ['xpool_9k_s24', 'xpool_9k_s0', 'xpool_9k_s1']
RUN_DIRS = [f'{DRIVE_ROOT}/runs/{e}' for e in EXPS]
for r in RUN_DIRS:
    assert os.path.exists(f'{r}/dump_model_best/dump_original.pt') and os.path.exists(f'{r}/model_best.pth'), r
OUT = f'{DRIVE_ROOT}/runs/pooling_budget'


In [ ]:
# ==== 1. Code: repo + pinned official X-Pool + deps ====
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip -q install -r requirements-colab.txt
!bash scripts/setup_xpool.sh
import sys; sys.path.insert(0, f"{REPO}/src")
!PYTHONPATH=src python -m pytest -q tests/ 2>&1 | tail -3


In [ ]:
# ==== 2a. QB-Norm querybank: 5,000 TRAINING captions embedded with each run's text encoder ====
for r in RUN_DIRS:
    if not os.path.exists(f'{r}/querybank_9k_5000.pt'):
        !PYTHONPATH=src python analysis/embed_querybank.py --ckpt {r}/model_best.pth --data_dir third_party/xpool/data/MSRVTT --out {r}/querybank_9k_5000.pt


In [ ]:
# ==== 2. Mean vs U-Pool vs X-Pool, attention sharpness, two-stage budgets, hubness, QB-Norm, fusion, timing ====
runs = ' '.join(RUN_DIRS)
!PYTHONPATH=src python analysis/pooling_budget.py --runs {runs} --out_dir {OUT}


In [ ]:
# ==== 3. Copy results (small files) to Drive results/pooling_budget/ ====
import shutil
dst = f'{DRIVE_ROOT}/results/pooling_budget'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(OUT, dst)
!ls -la {dst}
